## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import spacy
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict

# For visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# For skills extraction
import re
from nltk.tokenize import sent_tokenize
import nltk
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# Load spaCy model
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    print("Downloading spacy model...")
    import subprocess
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

print("All libraries imported successfully!")
print(f"spaCy model loaded: {nlp.meta['name']}")

All libraries imported successfully!


## 2. Load Resume Data

In [4]:
# Load resume data
df_resumes = pd.read_csv('data/Resume.csv')

print("Dataset shape:", df_resumes.shape)
print("\nColumn names:")
print(df_resumes.columns.tolist())

# Identify the resume column (text content)
if 'Resume_str' in df_resumes.columns:
    resume_column = 'Resume_str'
elif 'Resume_html' in df_resumes.columns:
    resume_column = 'Resume_html'
elif 'Resume' in df_resumes.columns:
    resume_column = 'Resume'
elif 'resume' in df_resumes.columns:
    resume_column = 'resume'
else:
    # Find the first column that contains text (string type)
    for col in df_resumes.columns:
        if df_resumes[col].dtype == 'object' and df_resumes[col].str.len().mean() > 50:
            resume_column = col
            break

print(f"Using column: {resume_column}")
print(f"\nSample resume (first 300 chars):\n{df_resumes[resume_column].iloc[0][:300]}...")


Dataset shape: (2484, 4)

Column names:
['ID', 'Resume_str', 'Resume_html', 'Category']
Using column: Resume_str

Sample resume (first 300 chars):
         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commit...


## 3. Define Skills Dictionary

In [ ]:
# Comprehensive skills dictionary with variations and synonyms
SKILLS_DICT = {
    'Programming Languages': {
        'skills': ['Python', 'Java', 'JavaScript', 'C++', 'C#', 'PHP', 'Ruby', 'Go', 'Rust', 'Kotlin',
                  'Swift', 'Objective-C', 'TypeScript', 'SQL', 'R', 'MATLAB', 'Scala', 'Perl', 'Bash',
                  'HTML', 'CSS', 'VB', 'COBOL', 'Groovy', 'Haskell'],
        'context_keywords': ['program', 'code', 'develop', 'language', 'script', 'write']
    },
    'Web Frameworks': {
        'skills': ['Django', 'Flask', 'FastAPI', 'Spring', 'Spring Boot', 'React', 'Vue', 'Angular',
                  'Node.js', 'Express', 'Laravel', 'Symfony', 'ASP.NET', 'Ruby on Rails', 'Meteor',
                  'Next.js', 'Nuxt', 'Svelte', 'Ember', 'Backbone'],
        'context_keywords': ['framework', 'frontend', 'backend', 'web', 'develop', 'full-stack']
    },
    'Databases': {
        'skills': ['MySQL', 'PostgreSQL', 'MongoDB', 'Redis', 'Cassandra', 'Oracle', 'SQLite',
                  'DynamoDB', 'Elasticsearch', 'Neo4j', 'CouchDB', 'Firestore', 'SQLServer',
                  'MariaDB', 'Apache Solr', 'InfluxDB', 'RavenDB'],
        'context_keywords': ['database', 'data', 'storage', 'query', 'sql', 'nosql']
    },
    'Data Science & ML': {
        'skills': ['NumPy', 'Pandas', 'Scikit-learn', 'TensorFlow', 'Keras', 'PyTorch', 'XGBoost',
                  'LightGBM', 'CatBoost', 'Statsmodels', 'NLTK', 'spaCy', 'Gensim', 'Plotly',
                  'Matplotlib', 'Seaborn', 'OpenCV', 'Jupyter', 'Google Analytics', 'Tableau',
                  'Power BI', 'Data Mining', 'Machine Learning', 'Deep Learning', 'NLP', 'Computer Vision',
                  'Apache Spark', 'Hadoop', 'Hive', 'Pig'],
        'context_keywords': ['model', 'analysis', 'prediction', 'learning', 'algorithm', 'data']
    },
    'Cloud & DevOps': {
        'skills': ['AWS', 'Azure', 'Google Cloud', 'Docker', 'Kubernetes', 'Jenkins', 'GitLab CI',
                  'GitHub Actions', 'Travis CI', 'CircleCI', 'Terraform', 'Ansible', 'Puppet',
                  'Chef', 'CloudFormation', 'Lambda', 'EC2', 'S3', 'GCP', 'Firebase', 'Heroku',
                  'DigitalOcean', 'OpenStack', 'Vagrant', 'Docker Compose'],
        'context_keywords': ['deploy', 'cloud', 'infrastructure', 'container', 'pipeline', 'devops']
    },
    'Version Control': {
        'skills': ['Git', 'GitHub', 'GitLab', 'Bitbucket', 'Mercurial', 'Subversion', 'SVN', 'Perforce'],
        'context_keywords': ['version', 'control', 'repository', 'commit', 'branch', 'source']
    },
    'Tools & Platforms': {
        'skills': ['Linux', 'Windows', 'macOS', 'Unix', 'Apache', 'Nginx', 'IIS', 'Tomcat',
                  'JBoss', 'WebLogic', 'Jira', 'Confluence', 'Slack', 'Agile', 'Scrum', 'Kanban',
                  'REST API', 'GraphQL', 'SOAP', 'XML', 'JSON', 'Protocol Buffers'],
        'context_keywords': ['platform', 'tool', 'environment', 'system', 'api', 'service']
    },
    'Testing': {
        'skills': ['JUnit', 'PyTest', 'Mocha', 'Jest', 'Jasmine', 'Selenium', 'Appium', 'Cypress',
                  'TestNG', 'Cucumber', 'BDD', 'TDD', 'Unit Testing', 'Integration Testing',
                  'End-to-End Testing', 'Performance Testing', 'Load Testing', 'Security Testing'],
        'context_keywords': ['test', 'qa', 'quality', 'verification', 'validation', 'automation']
    },
    'Soft Skills': {
        'skills': ['Leadership', 'Communication', 'Problem Solving', 'Critical Thinking', 'Teamwork',
                  'Project Management', 'Time Management', 'Adaptability', 'Creativity', 'Analytical Skills',
                  'Decision Making', 'Collaboration', 'Customer Service', 'Presentation', 'Documentation'],
        'context_keywords': ['lead', 'team', 'manage', 'communicate', 'solve', 'collaborate']
    }
}

# Create lookup dictionaries for lemma-based and fuzzy matching
all_skills_flat = {}
for category, data in SKILLS_DICT.items():
    for skill in data['skills']:
        all_skills_flat[skill.lower()] = {
            'original': skill,
            'category': category,
            'tokens': skill.lower().split()
        }

print(f"Total unique skills in dictionary: {len(all_skills_flat)}")
print(f"Categories: {list(SKILLS_DICT.keys())}")


Total unique skills in dictionary: 180
Categories: ['Programming Languages', 'Web Frameworks', 'Databases', 'Data Science & ML', 'Cloud & DevOps', 'Version Control', 'Tools & Platforms', 'Testing', 'Soft Skills']


## 4. Extract Skills from Resume Text

In [ ]:
def extract_skills_spacy_nlp(text, skills_dict, nlp, threshold=0.8):
    """
    Extract skills using spaCy NLP with context awareness and lemmatization.
    
    Args:
        text: Resume text
        skills_dict: Skills dictionary with context keywords
        nlp: spaCy language model
        threshold: Similarity threshold for fuzzy matching (0-1)
    
    Returns:
        List of found skills with context information
    """
    if pd.isna(text) or len(str(text).strip()) == 0:
        return []
    
    text = str(text)
    found_skills = []
    seen_skills = set()
    
    # Process text with spaCy
    doc = nlp(text)
    
    # Extract lemmas for better matching
    text_lemmas = set([token.lemma_.lower() for token in doc if not token.is_stop])
    text_tokens_lower = [token.text.lower() for token in doc if not token.is_punct]
    
    # For each skill category
    for category, data in skills_dict.items():
        skills = data['skills']
        context_keywords = data['context_keywords']
        
        # Check if category context is present
        category_context_present = any(
            lemma in text_lemmas 
            for keyword in context_keywords 
            for lemma in nlp(keyword)[0].lemma_.lower().split()
        )
        
        for skill in skills:
            skill_lower = skill.lower()
            skill_tokens = skill_lower.split()
            
            # Method 1: Exact match with word boundaries
            pattern = r'\b' + re.escape(skill_lower) + r'\b'
            if re.search(pattern, text.lower()):
                found_skills.append({
                    'skill': skill,
                    'category': category,
                    'original': skill_lower,
                    'method': 'exact_match',
                    'context_confidence': 1.0
                })
                seen_skills.add(skill_lower)
                continue
            
            # Method 2: Token-based matching with spaCy lemmatization
            skill_doc = nlp(skill)
            skill_lemmas = set([t.lemma_.lower() for t in skill_doc if not t.is_stop])
            
            # Check if all skill tokens appear (as lemmas) in the text
            matched_lemmas = skill_lemmas.intersection(text_lemmas)
            
            if len(skill_lemmas) > 0 and len(matched_lemmas) / len(skill_lemmas) >= threshold:
                # Calculate context strength
                context_strength = 1.0 if category_context_present else 0.7
                
                if skill_lower not in seen_skills:
                    found_skills.append({
                        'skill': skill,
                        'category': category,
                        'original': skill_lower,
                        'method': 'lemma_matching',
                        'context_confidence': context_strength,
                        'matched_tokens': len(matched_lemmas) / len(skill_lemmas)
                    })
                    seen_skills.add(skill_lower)
    
    # Sort by confidence (exact matches first, then by context strength)
    found_skills.sort(
        key=lambda x: (x['method'] == 'exact_match', x.get('context_confidence', 0.7)), 
        reverse=True
    )
    
    return found_skills


# Extract skills from all resumes
print("Extracting skills from all resumes using spaCy NLP...")
print("This may take a moment...")

df_resumes['skills'] = df_resumes[resume_column].apply(
    lambda x: extract_skills_spacy_nlp(x, SKILLS_DICT, nlp, threshold=0.75)
)
df_resumes['skill_count'] = df_resumes['skills'].apply(len)

print(f"\n✓ Skills extraction completed!")
print(f"Total skills found: {df_resumes['skill_count'].sum()}")
print(f"Average skills per resume: {df_resumes['skill_count'].mean():.2f}")
print(f"Min skills in a resume: {df_resumes['skill_count'].min()}")
print(f"Max skills in a resume: {df_resumes['skill_count'].max()}")

print(f"\nSample skills from first resume (top 15):")
for skill in df_resumes['skills'].iloc[0][:15]:
    method = f"[{skill['method']}]"
    confidence = f"{skill.get('context_confidence', 1.0):.2f}"
    print(f"  - {skill['skill']:30} | Category: {skill['category']:20} | {method:20} | Confidence: {confidence}")


Extracting skills from all resumes...

Skills extraction completed!
Total skills found: 9139
Average skills per resume: 3.68
Min skills in a resume: 0
Max skills in a resume: 33

Sample skills from first resume:
  - Swift                          | Category: Programming Languages
  - Leadership                     | Category: Soft Skills
  - Time Management                | Category: Soft Skills
  - Analytical Skills              | Category: Soft Skills
  - Customer Service               | Category: Soft Skills
  - Documentation                  | Category: Soft Skills


## 5. Aggregate Skills Analysis

In [ ]:
# Aggregate skills by category
skills_by_category = defaultdict(Counter)
all_found_skills = []
extraction_methods = Counter()

for skills_list in df_resumes['skills']:
    for skill_info in skills_list:
        skill = skill_info['skill']
        category = skill_info['category']
        method = skill_info.get('method', 'unknown')
        
        skills_by_category[category][skill] += 1
        all_found_skills.append(skill)
        extraction_methods[method] += 1

print("="*80)
print("SKILLS EXTRACTION SUMMARY")
print("="*80)

print(f"\nExtraction Methods Used:")
for method, count in extraction_methods.most_common():
    print(f"  - {method}: {count} skills")

print("\n" + "="*80)
print("SKILLS SUMMARY BY CATEGORY")
print("="*80)

for category in sorted(skills_by_category.keys()):
    skills = skills_by_category[category]
    total_count = sum(skills.values())
    unique_skills = len(skills)
    
    print(f"\n{category.upper()}")
    print("-" * 80)
    print(f"  Total mentions: {total_count} | Unique skills: {unique_skills}")
    print(f"  Top 10 skills:")
    
    top_10 = skills.most_common(10)
    for skill, count in top_10:
        percentage = (count / total_count) * 100
        print(f"    {skill:30} | Count: {count:3} | {percentage:5.2f}%")


SKILLS SUMMARY BY CATEGORY

CLOUD & DEVOPS
----------------------------------------------------------------------
  Total mentions: 214 | Unique skills: 13
  Top 10 skills:
    Chef                           | Count: 127 | 59.35%
    Aws                            | Count:  22 | 10.28%
    Lambda                         | Count:  19 |  8.88%
    Azure                          | Count:  11 |  5.14%
    Jenkins                        | Count:  10 |  4.67%
    Ec2                            | Count:   6 |  2.80%
    Puppet                         | Count:   5 |  2.34%
    S3                             | Count:   5 |  2.34%
    Gcp                            | Count:   4 |  1.87%
    Docker                         | Count:   2 |  0.93%

DATA SCIENCE & ML
----------------------------------------------------------------------
  Total mentions: 134 | Unique skills: 22
  Top 10 skills:
    Google Analytics               | Count:  52 | 38.81%
    Tableau                        | Count:  26 | 1

## 6. Visualizations

In [8]:
# Overall top 20 skills across all resumes
top_20_skills = Counter(all_found_skills).most_common(20)

fig1 = go.Figure()

fig1.add_trace(go.Bar(
    y=[skill[0] for skill in top_20_skills],
    x=[skill[1] for skill in top_20_skills],
    orientation='h',
    marker_color='steelblue',
    text=[skill[1] for skill in top_20_skills],
    textposition='auto'
))

fig1.update_layout(
    title="Top 20 Skills Found in All Resumes",
    xaxis_title="Frequency",
    yaxis_title="Skill",
    height=600,
    showlegend=False
)
fig1.show()

In [9]:
# Skills per category (pie chart)
category_counts = {category: sum(skills.values()) for category, skills in skills_by_category.items()}

fig2 = go.Figure(data=[go.Pie(
    labels=list(category_counts.keys()),
    values=list(category_counts.values()),
    hole=0.3
)])

fig2.update_layout(
    title="Distribution of Skills by Category",
    height=600
)
fig2.show()

## 7. Advanced NLP Analysis with spaCy


In [ ]:
def analyze_skill_context(resume_text, skill, nlp, context_window=3):
    """
    Analyze the context around a skill mention using spaCy.
    Returns sentences containing the skill and surrounding context.
    """
    doc = nlp(resume_text)
    sentences = list(doc.sents)
    
    contexts = []
    for sent in sentences:
        sent_text_lower = sent.text.lower()
        if skill.lower() in sent_text_lower:
            # Extract noun phrases and entities from the sentence
            noun_phrases = [chunk.text for chunk in doc[sent.start:sent.end].noun_chunks]
            entities = [(ent.text, ent.label_) for ent in doc[sent.start:sent.end].ents]
            
            contexts.append({
                'sentence': sent.text,
                'noun_phrases': noun_phrases,
                'entities': entities,
                'tokens': [token.text for token in sent if not token.is_punct]
            })
    
    return contexts


def extract_skill_relationships(resume_text, nlp):
    """
    Extract relationships between skills using dependency parsing.
    Identifies skills that are often mentioned together or in relation.
    """
    doc = nlp(resume_text)
    
    # Find all potential skill patterns through dependency parsing
    relationships = []
    
    for token in doc:
        # Look for coordination patterns (X and Y, X, Y, Z)
        if token.dep_ in ['cc', 'conj']:  # coordinating conjunction or conjunct
            head = token.head
            if head.text.lower() in [s.lower() for s in all_skills_flat.keys()]:
                if token.text.lower() in [s.lower() for s in all_skills_flat.keys()]:
                    relationships.append({
                        'skill1': head.text,
                        'skill2': token.text,
                        'relation': 'coordinated'
                    })
    
    return relationships

# Analyze skill context for first resume
print("="*80)
print("DETAILED SKILL CONTEXT ANALYSIS (First Resume)")
print("="*80)

first_resume = df_resumes[resume_column].iloc[0]
first_skills = df_resumes['skills'].iloc[0][:5]  # Top 5 skills

for skill_info in first_skills:
    skill = skill_info['skill']
    print(f"\n🔍 SKILL: {skill} (Category: {skill_info['category']})")
    print("-" * 80)
    
    contexts = analyze_skill_context(first_resume, skill, nlp)
    
    if contexts:
        print(f"Found in {len(contexts)} context(s):")
        for i, ctx in enumerate(contexts[:2], 1):  # Show first 2 contexts
            print(f"\n  Context {i}:")
            print(f"  📝 {ctx['sentence'][:100]}...")
            if ctx['noun_phrases']:
                print(f"  📌 Related terms: {', '.join(ctx['noun_phrases'][:3])}")
    else:
        print("  (No specific context found)")


## 8. Skill Co-occurrence Analysis


In [ ]:
# Analyze skill co-occurrence patterns
from itertools import combinations

skill_cooccurrence = Counter()

# For each resume, find skills that appear together
for skills_list in df_resumes['skills']:
    skill_names = [s['skill'] for s in skills_list]
    
    # Generate all pairs of skills in this resume
    for skill1, skill2 in combinations(sorted(skill_names), 2):
        pair = tuple(sorted([skill1, skill2]))
        skill_cooccurrence[pair] += 1

# Get top co-occurring skill pairs
top_cooccurrences = skill_cooccurrence.most_common(20)

print("="*80)
print("TOP 20 SKILL CO-OCCURRENCE PATTERNS")
print("="*80)
print(f"\n(Skills that frequently appear together in resumes)\n")

for i, (skill_pair, count) in enumerate(top_cooccurrences, 1):
    skill1, skill2 = skill_pair
    print(f"{i:2}. {skill1:25} + {skill2:25} | Appears together: {count} resumes")

# Create co-occurrence network visualization
print("\n" + "="*80)
print("SKILL CLUSTER ANALYSIS")
print("="*80)

# Group skills by category co-occurrence
category_cooccurrence = Counter()

for skills_list in df_resumes['skills']:
    categories = set([s['category'] for s in skills_list])
    
    for cat1, cat2 in combinations(sorted(categories), 2):
        pair = tuple(sorted([cat1, cat2]))
        category_cooccurrence[pair] += 1

print("\nTop category combinations in resumes:")
for (cat1, cat2), count in category_cooccurrence.most_common(10):
    print(f"  {cat1:25} + {cat2:25} | {count} resumes")
